# 🌍 Clustering de Terremotos Severos (1995–2023)
### Projeto Final — Análise Preditiva | Equipe 13: Catástrofes Ambientais

**Integrantes:** Caio Monteiro  
**Dataset:** Global Significant Earthquake Database (Kaggle) — 1.000 eventos sísmicos de magnitude ≥ 6,5  
**Algoritmo principal:** K-Means | **Algoritmo complementar:** validação por Silhouette Score  
**Objetivo:** identificar perfis de terremotos severos para subsidiar estratégias de preparação e resposta a desastres.


## 1. Problemática

Terremotos severos (magnitude ≥ 6,5) figuram entre as catástrofes naturais de maior potencial destrutivo: podem gerar tsunamis, colapso de infraestrutura, mortes em massa e perdas econômicas de escala bilionária. Entre 1995 e 2023, mais de 1.000 eventos dessa magnitude foram registrados globalmente.

**Problema central:** embora todos esses eventos sejam "severos" por definição, existe uma grande heterogeneidade entre eles — um sismo de M 6,5 a 500 km de profundidade tem impacto radicalmente diferente de um M 8,0 raso numa região costeira densamente povoada.

**Pergunta de negócio:** *É possível agrupar esses terremotos em perfis distintos que permitam priorizar estratégias de alerta, preparação e resposta emergencial?*

Utilizaremos **clustering não supervisionado** para segmentar os eventos com base em características sísmicas objetivas — profundidade, magnitude, significância, impacto sentido pela população e geração de tsunamis — e assim criar perfis acionáveis para gestores de riscos e defesa civil.


## 2. Carregamento das Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100


## 3. Carregamento e Inspeção Inicial dos Dados

O dataset foi obtido no Kaggle ([Global Significant Earthquake Database](https://www.kaggle.com/datasets/alessandrolobello/the-ultimate-earthquake-dataset-from-1990-2023)) e reúne 1.000 eventos sísmicos de magnitude ≥ 6,5 registrados entre 1995 e 2023. Cada linha representa um terremoto individual, com 19 atributos técnicos coletados pela rede de monitoramento USGS (United States Geological Survey).

Abaixo carregamos o arquivo e realizamos uma inspeção inicial para entender a estrutura geral dos dados.


In [ ]:
df = pd.read_csv('earthquake_1995-2023.csv')
print(f"Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")
df.head()


In [ ]:
df.info()


In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])


### Dicionário das Variáveis

| Coluna | Tipo | Descrição |
|---|---|---|
| `title` | str | Descrição textual do evento |
| `magnitude` | float | Magnitude do terremoto (escala Richter/Momento) |
| `date_time` | str | Data e hora do evento (UTC) |
| `cdi` | int | *Community Internet Intensity* — intensidade máxima reportada por cidadãos (0–9) |
| `mmi` | int | *Modified Mercalli Intensity* — intensidade instrumental estimada (0–12) |
| `alert` | str | Nível de alerta PAGER: green / yellow / orange / red |
| `tsunami` | int | Flag binária: 1 = gerou tsunami, 0 = não gerou |
| `sig` | int | Índice de significância USGS (função de magnitude, intensidade e impacto) |
| `net` | str | Rede sismológica que reportou o evento |
| `nst` | int | Número de estações sísmicas que detectaram o evento |
| `dmin` | float | Distância mínima (graus) à estação sismológica mais próxima |
| `gap` | float | Maior lacuna azimutal entre estações detectoras (graus) |
| `magType` | str | Método de cálculo da magnitude |
| `depth` | float | Profundidade do hipocentro (km) |
| `latitude` / `longitude` | float | Coordenadas geográficas do epicentro |
| `location` | str | Nome do local mais próximo |
| `continent` / `country` | str | Continente e país do epicentro |


## 4. Pré-processamento

### 4.1 Tratamento da coluna de data

Convertemos `date_time` para o tipo `datetime` e extraímos `year` e `month` para facilitar análises temporais na EDA.


In [ ]:
df['date_time'] = pd.to_datetime(df['date_time'], format='%d-%m-%Y %H:%M')
df['year']  = df['date_time'].dt.year
df['month'] = df['date_time'].dt.month
print("Período coberto:", df['date_time'].min().date(), "→", df['date_time'].max().date())


### 4.2 Análise de Dados Faltantes

Antes de qualquer modelagem, é essencial entender quais colunas possuem valores ausentes e decidir como tratá-los.


In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'Nulos': missing, '% Faltante': missing_pct})
missing_df[missing_df['Nulos'] > 0]


In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Mapa de Dados Faltantes', fontsize=14)
plt.tight_layout()
plt.show()


**Observações:**
- `continent` (71,6%) e `alert` (55,1%) têm alta proporção de nulos — **não serão usadas como features de clusterização**, mas servirão para análise e interpretação dos clusters.
- `country` (34,9%) e `location` (0,6%) têm o mesmo tratamento acima.
- As 9 features numéricas escolhidas para o modelo (`magnitude`, `depth`, `sig`, `cdi`, `mmi`, `tsunami`, `nst`, `dmin`, `gap`) **não possuem valores faltantes**, garantindo integridade total para a clusterização.


## 5. Análise Exploratória dos Dados (EDA)

### 5.1 Distribuição das Variáveis Numéricas


In [ ]:
features_num = ['magnitude', 'depth', 'sig', 'cdi', 'mmi', 'tsunami', 'nst', 'dmin', 'gap']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(features_num):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frequência')

plt.suptitle('Distribuição das Features de Clusterização', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()


**Destaques da distribuição:**
- **magnitude**: concentrada entre 6,5 e 7,0 (63% dos eventos), com cauda longa até M 9,1 (Japão 2011).
- **depth**: fortemente assimétrica à direita — a maioria dos eventos é rasa (< 100 km), mas existem sismos profundos até 670 km.
- **sig**: varia de 650 a 2.910; eventos de alta significância são raros mas existentes.
- **tsunami**: variável binária — 32,5% dos eventos geraram tsunami.
- **nst**: número de estações varia muito, refletindo a cobertura geográfica desigual da rede de monitoramento.


### 5.2 Matriz de Correlação

In [ ]:
mask = np.triu(np.ones_like(df[features_num].corr(), dtype=bool))
plt.figure(figsize=(11, 9))
sns.heatmap(
    df[features_num].corr(),
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5,
    vmin=-1, vmax=1
)
plt.title('Matriz de Correlação — Features de Clusterização', fontsize=13)
plt.tight_layout()
plt.show()


**Principais relações identificadas:**
- `magnitude` e `sig` têm correlação positiva forte (~0,85): sismos mais intensos são naturalmente mais significativos.
- `cdi` e `mmi` têm alta correlação (~0,80): ambas medem intensidade, porém por métodos diferentes (cidadãos vs. instrumental).
- `depth` tem correlação negativa moderada com `cdi` e `mmi`: terremotos mais profundos tendem a ser menos sentidos na superfície.
- `tsunami` tem correlação positiva moderada com `magnitude` (~0,42): megaterremotos rasos oceânicos são os principais geradores de tsunamis.


### 5.3 Distribuição Temporal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Eventos por ano
events_year = df.groupby('year').size()
events_year.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Número de Terremotos Severos por Ano', fontsize=13)
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Quantidade')
axes[0].tick_params(axis='x', rotation=45)

# Magnitude média por ano
mag_year = df.groupby('year')['magnitude'].mean()
axes[1].plot(mag_year.index, mag_year.values, marker='o', color='tomato', linewidth=2)
axes[1].set_title('Magnitude Média por Ano', fontsize=13)
axes[1].set_xlabel('Ano')
axes[1].set_ylabel('Magnitude Média')
axes[1].axhline(mag_year.mean(), color='gray', linestyle='--', alpha=0.7, label=f'Média geral: {mag_year.mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()


O volume de registros é relativamente estável ao longo do período (25–53 eventos/ano), sem tendência clara de aumento. O pico de 2013–2016 pode refletir tanto aumento real de atividade sísmica quanto melhoria na cobertura da rede de monitoramento. A magnitude média permanece estável em torno de 6,9, confirmando que o dataset representa um recorte consistente de eventos severos.


### 5.4 Distribuição Geográfica

In [ ]:
plt.figure(figsize=(16, 7))
scatter = plt.scatter(
    df['longitude'], df['latitude'],
    c=df['magnitude'], cmap='YlOrRd',
    alpha=0.6, s=30, edgecolors='none'
)
plt.colorbar(scatter, label='Magnitude')
plt.title('Distribuição Geográfica dos Terremotos Severos (1995–2023)', fontsize=13)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
plt.tight_layout()
plt.show()


O mapa revela claramente o **Anel de Fogo do Pacífico** como principal zona sísmica global, concentrando a grande maioria dos eventos. Destaca-se também a faixa himalaia (placa Indo-Australiana) e as ilhas do Pacífico Sul. Regiões do interior dos continentes são praticamente ausentes, confirmando que terremotos severos se concentram nas bordas de placas tectônicas.


### 5.5 Perfil de Tsunamis e Alertas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tsunami por faixa de magnitude
df['mag_bin'] = pd.cut(df['magnitude'], bins=[6.4, 6.9, 7.4, 7.9, 8.4, 9.2],
                        labels=['6,5–6,9', '7,0–7,4', '7,5–7,9', '8,0–8,4', '8,5+'])
tsunami_mag = df.groupby('mag_bin', observed=True)['tsunami'].mean() * 100
tsunami_mag.plot(kind='bar', ax=axes[0], color=['#4CAF50','#FFC107','#FF9800','#F44336','#9C27B0'],
                  edgecolor='white')
axes[0].set_title('Taxa de Geração de Tsunami por Faixa de Magnitude (%)', fontsize=12)
axes[0].set_xlabel('Faixa de Magnitude')
axes[0].set_ylabel('% com Tsunami')
axes[0].tick_params(axis='x', rotation=0)

# Distribuição dos alertas (sem nulos)
alert_counts = df['alert'].value_counts()
cores_alert = {'green': '#4CAF50', 'yellow': '#FFC107', 'orange': '#FF9800', 'red': '#F44336'}
colors = [cores_alert[a] for a in alert_counts.index]
alert_counts.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Distribuição dos Níveis de Alerta PAGER (eventos com alerta)', fontsize=12)
axes[1].set_xlabel('Nível de Alerta')
axes[1].set_ylabel('Quantidade')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


A taxa de geração de tsunami cresce dramaticamente com a magnitude: enquanto apenas ~20% dos eventos M 6,5–6,9 geram tsunamis, a taxa sobe para mais de 60% acima de M 8,0. Quanto aos alertas, a grande maioria dos eventos classificados se enquadra no nível "green" (baixo impacto humano esperado), mas os 13 alertas "red" representam os eventos mais devastadores da série histórica.


## 6. Preparação dos Dados para Clusterização

### 6.1 Seleção de Features

Para a clusterização, selecionamos 9 variáveis numéricas que caracterizam o **perfil sísmico e de impacto** de cada evento, excluindo colunas com alta proporção de nulos e variáveis textuais/identificadoras:

| Feature | Justificativa |
|---|---|
| `magnitude` | Força intrínseca do evento |
| `depth` | Profundidade do hipocentro — afeta propagação e impacto |
| `sig` | Índice de significância USGS — proxy de relevância global |
| `cdi` | Impacto sentido pela população |
| `mmi` | Intensidade instrumental estimada |
| `tsunami` | Geração de tsunami — risco secundário crítico |
| `nst` | Cobertura de monitoramento (qualidade do dado) |
| `dmin` | Proximidade à estação mais próxima |
| `gap` | Lacuna azimutal — confiabilidade da localização |


In [ ]:
features = ['magnitude', 'depth', 'sig', 'cdi', 'mmi', 'tsunami', 'nst', 'dmin', 'gap']
X_raw = df[features].copy()
print(f"Dataset para clusterização: {X_raw.shape}")
print(f"Nulos: {X_raw.isnull().sum().sum()} (nenhum)")
X_raw.describe().round(2)


### 6.2 Normalização com StandardScaler

K-Means utiliza distância euclidiana para formar os clusters — portanto é **sensível à escala das variáveis**. Sem normalização, variáveis com valores grandes (como `sig` que chega a 2.910) dominariam completamente o cálculo de distâncias sobre variáveis menores (como `magnitude` que varia entre 6,5 e 9,1).

O `StandardScaler` transforma cada variável para média 0 e desvio padrão 1, garantindo que todas contribuam igualmente para a formação dos clusters.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Confirmar normalização
pd.DataFrame(X_scaled, columns=features).describe().round(4)


## 7. Construção dos Clusters

### 7.1 Justificativa do Algoritmo: K-Means

**Por que K-Means?**
- Dataset de tamanho médio (1.000 registros) com variáveis numéricas contínuas: cenário ideal para K-Means.
- Alta interpretabilidade: cada cluster tem um centroide claro que resume o perfil médio do grupo.
- Eficiência computacional: converge rapidamente mesmo com múltiplas inicializações.
- Amplamente utilizado em análises sísmicas e de risco natural na literatura científica.

**Limitação reconhecida:** K-Means assume clusters esféricos e é sensível a outliers. Para mitigar, utilizamos `n_init=10` (10 inicializações aleatórias diferentes) e a estratégia `k-means++` para inicialização inteligente dos centroides.

### 7.2 Método do Cotovelo + Silhouette Score para escolha de K


In [ ]:
inertias    = []
silhouettes = []
K_range     = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Cotovelo
axes[0].plot(K_range, inertias, marker='o', color='steelblue', linewidth=2, markersize=8)
axes[0].set_title('Método do Cotovelo (Inércia × K)', fontsize=13)
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inércia (WCSS)')
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='K escolhido = 4')
axes[0].legend()

# Silhouette
axes[1].plot(K_range, silhouettes, marker='s', color='tomato', linewidth=2, markersize=8)
axes[1].set_title('Silhouette Score × K', fontsize=13)
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].axvline(x=4, color='red', linestyle='--', alpha=0.7, label=f'K=4 → score={max(silhouettes):.4f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nTabela de resultados:")
res = pd.DataFrame({'K': list(K_range), 'Inércia': inertias, 'Silhouette': [round(s, 4) for s in silhouettes]})
print(res.to_string(index=False))


**K = 4 foi escolhido** com base em dois critérios complementares:
1. **Método do Cotovelo:** a inércia apresenta uma inflexão visível em K=4, após a qual os ganhos de redução se tornam marginais.
2. **Silhouette Score:** K=4 apresenta o maior coeficiente (0,2781) entre todos os valores testados, indicando a melhor separação relativa entre os clusters.

Um Silhouette Score de ~0,28 é esperado para dados sísmicos naturais, pois os perfis de terremotos formam gradientes contínuos (não há fronteiras perfeitamente nítidas entre tipos de eventos) — ainda assim, os perfis encontrados são clara e interpretativamente distintos.


### 7.3 Treinamento do Modelo Final (K=4)

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

print("Distribuição dos clusters:")
cluster_counts = df['cluster'].value_counts().sort_index()
for c, n in cluster_counts.items():
    print(f"  Cluster {c}: {n} eventos ({n/len(df)*100:.1f}%)")


## 8. Avaliação dos Clusters

Utilizamos três métricas complementares para avaliar a qualidade dos agrupamentos:

| Métrica | O que mede | Interpretação |
|---|---|---|
| **Silhouette Score** | Coesão interna vs. separação externa | Varia de -1 a 1; quanto maior, melhor |
| **Davies-Bouldin Index** | Similaridade média entre clusters | Quanto menor, melhor (0 = perfeito) |
| **Calinski-Harabasz Index** | Razão entre dispersão inter e intra-cluster | Quanto maior, melhor |


In [ ]:
sil = silhouette_score(X_scaled, df['cluster'])
db  = davies_bouldin_score(X_scaled, df['cluster'])
ch  = calinski_harabasz_score(X_scaled, df['cluster'])

print("=" * 45)
print(f"  Silhouette Score     : {sil:.4f}")
print(f"  Davies-Bouldin Index : {db:.4f}")
print(f"  Calinski-Harabasz   : {ch:.2f}")
print("=" * 45)


**Interpretação dos resultados:**
- **Silhouette = 0,2781:** coeficiente positivo confirma que a maioria dos pontos está razoavelmente mais próxima do seu próprio cluster do que dos demais. O valor moderado (abaixo de 0,5) reflete a natureza contínua dos dados sísmicos — não há "clusters perfeitos" na geofísica, pois os eventos formam espectros contínuos.
- **Davies-Bouldin = 1,34:** indica separação razoável entre os clusters (referência: DBI < 1,5 é considerado aceitável para dados reais).
- **Calinski-Harabasz = 237,1:** valor relativamente alto confirma que os clusters são densos internamente e bem separados entre si.

Em conjunto, as métricas validam que K=4 produz agrupamentos meaningful e interpretáveis.


### 8.1 Visualização com PCA (redução para 2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print(f"Variância explicada: PC1={explained[0]:.1%} | PC2={explained[1]:.1%} | Total={sum(explained):.1%}")

plt.figure(figsize=(12, 7))
cores = {0: '#2196F3', 1: '#F44336', 2: '#9C27B0', 3: '#4CAF50'}
nomes = {
    0: 'Cluster 0 — Costeiros c/ Tsunami',
    1: 'Cluster 1 — Megaterremotos Destrutivos',
    2: 'Cluster 2 — Terremotos Profundos',
    3: 'Cluster 3 — Rasos de Baixo Impacto'
}

for c in range(4):
    mask_c = df['cluster'] == c
    plt.scatter(X_pca[mask_c, 0], X_pca[mask_c, 1],
                c=cores[c], label=nomes[c], alpha=0.6, s=40, edgecolors='none')

# Centroides projetados
centroides_pca = pca.transform(kmeans.cluster_centers_)
for c in range(4):
    plt.scatter(centroides_pca[c, 0], centroides_pca[c, 1],
                c=cores[c], s=250, marker='*', edgecolors='black', linewidths=1.5, zorder=5)

plt.title('Visualização dos Clusters — Projeção PCA 2D\n(★ = centroide)', fontsize=13)
plt.xlabel(f'Componente Principal 1 ({explained[0]:.1%} da variância)')
plt.ylabel(f'Componente Principal 2 ({explained[1]:.1%} da variância)')
plt.legend(loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.show()


## 9. Descrição Detalhada dos Clusters

### 9.1 Perfil Estatístico dos Clusters


In [ ]:
# Estatísticas por cluster
perfil = df.groupby('cluster')[features].mean().round(2)
perfil.index = [
    'Cluster 0 — Costeiros c/ Tsunami',
    'Cluster 1 — Megaterremotos Destrutivos',
    'Cluster 2 — Terremotos Profundos',
    'Cluster 3 — Rasos de Baixo Impacto'
]
perfil.T.style.background_gradient(cmap='RdYlGn', axis=1)


In [ ]:
# Tamanho e distribuição de alertas por cluster
print("Tamanho dos clusters:")
print(df.groupby('cluster').size().rename('n_eventos'))
print()
print("Distribuição de alertas por cluster:")
alert_cluster = pd.crosstab(df['cluster'], df['alert'])
alert_cluster.index = ['C0', 'C1', 'C2', 'C3']
print(alert_cluster)
print()
print("Taxa de tsunami por cluster:")
print(df.groupby('cluster')['tsunami'].mean().round(3) * 100, "%")


### 9.2 Radar Chart dos Perfis (Normalizado)


In [ ]:
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

# Normalizar perfis para o radar (0-1)
perfil_norm = df.groupby('cluster')[features].mean()
perfil_norm = (perfil_norm - perfil_norm.min()) / (perfil_norm.max() - perfil_norm.min())

categories = features
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(2, 2, figsize=(14, 12), subplot_kw=dict(polar=True))
axes = axes.flatten()

cores_radar = ['#2196F3', '#F44336', '#9C27B0', '#4CAF50']
nomes_cluster = [
    'Cluster 0\nCosteiros c/ Tsunami',
    'Cluster 1\nMegaterremotos Destrutivos',
    'Cluster 2\nTerremotos Profundos',
    'Cluster 3\nRasos de Baixo Impacto'
]

for idx in range(4):
    ax = axes[idx]
    values = perfil_norm.iloc[idx].values.tolist()
    values += values[:1]

    ax.plot(angles, values, color=cores_radar[idx], linewidth=2)
    ax.fill(angles, values, color=cores_radar[idx], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=9)
    ax.set_ylim(0, 1)
    ax.set_title(nomes_cluster[idx], size=11, fontweight='bold', pad=15)
    ax.grid(True, alpha=0.4)

plt.suptitle('Radar Chart dos Perfis de Cluster (valores normalizados)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


### 9.3 Boxplots por Cluster — Variáveis-Chave


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

key_vars  = ['magnitude', 'depth', 'sig', 'cdi', 'mmi', 'tsunami']
cores_box = ['#2196F3', '#F44336', '#9C27B0', '#4CAF50']

for i, var in enumerate(key_vars):
    data_by_cluster = [df[df['cluster'] == c][var].values for c in range(4)]
    bp = axes[i].boxplot(data_by_cluster, patch_artist=True,
                          medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], cores_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[i].set_title(f'Distribuição de {var} por Cluster', fontsize=12)
    axes[i].set_xticklabels(['C0', 'C1', 'C2', 'C3'])
    axes[i].set_xlabel('Cluster')
    axes[i].set_ylabel(var)

plt.suptitle('Variáveis-Chave por Cluster', fontsize=14)
plt.tight_layout()
plt.show()


### 9.4 Descrição Narrativa dos Clusters

---

#### 🔵 Cluster 0 — *Terremotos Costeiros com Alto Risco de Tsunami*
**300 eventos (30,0%)**

Este é o grupo de **maior risco de tsunami** do dataset: 81% dos seus eventos geraram ondas de tsunami. São sismos de magnitude moderada-alta (média 6,81), rasos (profundidade média 46,8 km) e com alta intensidade sentida pela população (CDI médio 5,1). O índice de significância médio (772) é moderado, mas o alto potencial de geração de tsunamis os torna especialmente perigosos para populações costeiras.

Predominam na bacia do Pacífico e ao longo de zonas de subducção costeiras. O sistema de alerta PAGER classifica a maioria em nível "green", mas isso reflete impacto humano imediato — o risco de tsunami secundário pode ser catastrófico.

**Perfil sísmico:** raso · magnitude moderada · alto tsunami · impacto humano intermediário

---

#### 🔴 Cluster 1 — *Megaterremotos de Alta Destruição*
**131 eventos (13,1%)**

O cluster mais destruidor e, ao mesmo tempo, mais raro. Concentra os eventos de **maior magnitude** (média 7,48, máx 9,1 — o Tohoku 2011), **maior significância** (sig médio 1.436), **maior intensidade instrumental** (MMI 7,5) e **maior impacto sentido** (CDI 7,56). São sismos rasos (38 km em média), detectados por grande número de estações (NST médio 249).

Neste cluster estão os eventos responsáveis pelas maiores perdas de vidas do período: Sumatra 2004 (M 9,1), Haiti 2010 e Japão 2011. A distribuição de alertas PAGER confirma: 13 alertas "red" e 16 "orange" estão concentrados aqui.

**Perfil sísmico:** muito raso · altíssima magnitude · máxima destruição · muito bem monitorado

---

#### 🟣 Cluster 2 — *Terremotos Profundos*
**57 eventos (5,7%)**

Grupo caracterizado pela **profundidade excepcional** do hipocentro: média de 559 km, com eventos chegando a 670 km — zona da descontinuidade de Bullen, próxima à transição manto superior-inferior. Apesar da magnitude moderada (6,95 em média), a profundidade extrema atenua drasticamente a propagação das ondas sísmicas, resultando em CDI (2,96) e MMI (3,28) muito menores do que se esperaria para essa magnitude.

A taxa de tsunami (47%) pode parecer alta, mas resulta de poucos eventos raros em que a propagação de ondas de longa duração ocasionalmente altera a coluna d'água — não é o mecanismo usual de tsunami raso.

**Perfil sísmico:** profundidade extrema · impacto de superfície atenuado · baixa destruição direta

---

#### 🟢 Cluster 3 — *Terremotos Rasos Remotos de Baixo Impacto Humano*
**512 eventos (51,2%)**

O cluster mais numeroso. São sismos rasos (46 km em média) de magnitude moderada (6,88), com características instrumentais curiosas: **menor CDI** (1,79) do dataset, apesar de MMI instrumental razoável (6,22). Isso revela sismos em **regiões remotas** — oceânicas ou despovoadas — onde poucas pessoas os sentem diretamente. A taxa de tsunami é quase nula (3%).

O `dmin` muito baixo (0,04°) e NST alto (293 estações) indicam excelente cobertura de monitoramento — esses sismos ocorrem perto de redes densas de sensores mas longe de populações. A maioria tem alerta "green" (32 eventos) com apenas 1 alerta "orange".

**Perfil sísmico:** raso · bem monitorado · remoto/oceânico · mínimo impacto humano


### 9.5 Distribuição Geográfica por Cluster

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

cores_geo  = {0: '#2196F3', 1: '#F44336', 2: '#9C27B0', 3: '#4CAF50'}
nomes_geo  = {
    0: 'Cluster 0 — Costeiros c/ Tsunami',
    1: 'Cluster 1 — Megaterremotos Destrutivos',
    2: 'Cluster 2 — Terremotos Profundos',
    3: 'Cluster 3 — Rasos de Baixo Impacto'
}

for c in range(4):
    sub = df[df['cluster'] == c]
    axes[c].scatter(sub['longitude'], sub['latitude'],
                    c=cores_geo[c], alpha=0.5, s=25, edgecolors='none')
    axes[c].set_title(nomes_geo[c], fontsize=11, fontweight='bold')
    axes[c].set_xlabel('Longitude')
    axes[c].set_ylabel('Latitude')
    axes[c].axhline(0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    axes[c].set_xlim(-180, 180)
    axes[c].set_ylim(-70, 75)
    axes[c].text(0.02, 0.95, f'n = {len(sub)}', transform=axes[c].transAxes,
                  fontsize=10, verticalalignment='top',
                  bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.suptitle('Distribuição Geográfica por Cluster', fontsize=14)
plt.tight_layout()
plt.show()


## 10. Conclusão e Recomendações

### Síntese dos Resultados

A aplicação do K-Means com K=4 ao dataset de terremotos severos (1995–2023) produziu **quatro perfis sísmicos distintos e interpretáveis**, validados por Silhouette Score (0,278), Davies-Bouldin Index (1,34) e Calinski-Harabasz (237). A tabela abaixo resume os clusters:

| Cluster | Nome | n | Risco Principal | Ação Prioritária |
|---|---|---|---|---|
| **C0** | Costeiros c/ Tsunami | 300 (30%) | Tsunami | Sistemas de alerta precoce costeiros |
| **C1** | Megaterremotos Destrutivos | 131 (13%) | Destruição estrutural + tsunami | Engenharia antissísmica + plano de evacuação |
| **C2** | Terremotos Profundos | 57 (6%) | Baixo (monitoramento) | Estudo científico; baixa urgência operacional |
| **C3** | Rasos Remotos | 512 (51%) | Mínimo | Monitoramento de rotina |

### Recomendações Estratégicas

1. **Priorizar zonas do Cluster 0** para instalação e manutenção de sistemas de alerta de tsunamis — 81% de taxa de geração justifica investimento imediato em buoys oceânicas e sirenas costeiras no Pacífico.

2. **Reforço estrutural urgente** em regiões historicamente afetadas pelo Cluster 1 — Japão, Indonésia e Chile concentram a maior parte dos megaterremotos. Normas de construção antissísmica e simulacros periódicos são indispensáveis.

3. **Cluster 2 como laboratório científico** — apesar do baixo impacto imediato, sismos profundos fornecem informação única sobre a estrutura interna do planeta; investir em pesquisa nesse grupo tem retorno científico e de previsão de risco de longo prazo.

4. **Monitoramento de Cluster 3** pode ser otimizado (custo/benefício) — dado o baixo impacto humano desses eventos, os recursos de resposta emergencial podem ser racionalizados para os Clusters 0 e 1.

5. **Melhoria dos dados faltantes** em `continent` e `alert` é crítica para análises futuras — 71,6% de nulos em `continent` limita análises regionais e deve ser corrigido com enriquecimento geoespacial automatizado.


## Referências

- **Dataset:** Lobello, Alessandro. *The Ultimate Earthquake Dataset (1990–2023)*. Kaggle, 2023. Disponível em: https://www.kaggle.com/datasets/alessandrolobello/the-ultimate-earthquake-dataset-from-1990-2023
- **USGS Earthquake Hazards Program.** Earthquake Catalog. https://earthquake.usgs.gov/earthquakes/search/
- **Scikit-learn:** Pedregosa et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.
- **Rousseeuw, P.J.** (1987). Silhouettes: A graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics*, 20, 53–65.
- **Davies, D.L. & Bouldin, D.W.** (1979). A Cluster Separation Measure. *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 1(2), 224–227.
